# ☕ Decorator Design Pattern — Hinglish mein samjho (Beginner Friendly)

Is notebook mein hum **Decorator Design Pattern** ko step-by-step, Hinglish comments ke saath samjhenge.
Har code cell ke andar ek **multiline comment (docstring)** hai jisme concept Hinglish mein explain kiya gaya hai.

Pehle dekhenge **bina pattern ke** kya problem aati hai, phir **Decorator Pattern** se use solve karenge.
Aakhir mein ek **Practice Question** aur uska **Coding Solution** bhi diya gaya hai taaki tum khud practice kar sako.

> Note: Ye humara pehla **STRUCTURAL** pattern hai (Creational se alag category) — iska focus hai
> "objects ko ek DOOSRE SE kaise JODA jaaye" taaki naya behavior mile.

In [1]:
"""
DECORATOR DESIGN PATTERN — KYA HAI?
========================================

Decorator ek STRUCTURAL design pattern hai jo kisi OBJECT mein
NAYA BEHAVIOR (feature) DYNAMICALLY (runtime par) ADD karne deta
hai -- WO BHI bina uski ORIGINAL CLASS ko CHHUE!

SOCHO EK EXAMPLE:
--------------------
Ek Coffee Shop hai jahan Plain Coffee milti hai. Customers chahte
hain Milk, Sugar, Whipped Cream jaise ADD-ONS. Agar hum HAR
COMBINATION ke liye ALAG CLASS banayein:
    - PlainCoffee
    - CoffeeWithMilk
    - CoffeeWithSugar
    - CoffeeWithMilkAndSugar
    - CoffeeWithMilkAndSugarAndCream
    - ... (aur bhi combinations)

To classes ki SANKHYA EXPONENTIALLY badh jaayegi! (ise "class
explosion" kehte hain)

DECORATOR isko solve karta hai:
    -> Ek "base" object banao (PlainCoffee)
    -> Use "WRAP" karo Decorator classes se (MilkDecorator,
       SugarDecorator)
    -> Har Decorator apna THODA SA behavior add karta hai aur
       BAAKI kaam ANDAR wale object ko DELEGATE (forward) kar
       deta hai
    -> Decorators ko JITNI CHAHO utni baar, KISI BHI ORDER mein
       COMBINE kar sakte ho -- RUNTIME par!

KAISE KAAM KARTA HAI (MECHANISM):
--------------------------------------
1. Ek COMMON INTERFACE banao jise Component aur Decorator DONO
   follow karein (e.g. Coffee -- get_cost(), get_description())
2. Concrete Component banao (PlainCoffee) -- BASE object
3. Ek abstract Decorator banao jo Component ko "WRAP" karta hai
   (andar ek Component object STORE karta hai)
4. Concrete Decorators banao (MilkDecorator, SugarDecorator) jo
   apna EXTRA behavior add karke, BAAKI ANDAR wale object ko
   DELEGATE kar dete hain

MAIN COMPONENTS:
--------------------
1. Component Interface   -> Common operations define karta hai
                            (e.g. Coffee)
2. Concrete Component      -> Core/base object (e.g. PlainCoffee)
3. Decorator                -> Abstract wrapper, Component ka
                            reference rakhta hai
4. Concrete Decorator        -> Specific decorator jo behavior
                            add karta hai (e.g. MilkDecorator)

REAL LIFE EXAMPLES:
-----------------------
- Coffee Shop: base coffee + milk/sugar/cream add-ons
- Video Streaming: video + subtitles/audio-enhancements/language
- Text Processing: plain text + bold/italic/underline
- Java I/O Streams: FileInputStream + BufferedInputStream +
  DataInputStream (layers wrap ek doosre ko)

FAYDE (ADVANTAGES):
------------------------
- RUNTIME par behavior ADD/REMOVE kar sakte ho -- FLEXIBLE hai
- SINGLE RESPONSIBILITY PRINCIPLE follow hota hai -- har decorator
  EK CHHOTA SA kaam karta hai
- Bade, saari features combine karne wale SUBCLASSES nahi
  banane padte -- "class explosion" se BACHTE hain

NUKSAAN (DISADVANTAGES):
----------------------------
- Bahut saari CHHOTI CHHOTI decorator classes ban jaati hain --
  system COMPLEX ho sakta hai
- Debugging MUSHKIL ho sakti hai -- pata karna padta hai ki
  "WRAPPED" object ke ANDAR KITNI LAYERS hain
- OVERUSE karne se code MAINTAIN karna MUSHKIL ho sakta hai
"""
print("Concept samajh liya! Ab neeche code mein implement karke dekhte hain.")


Concept samajh liya! Ab neeche code mein implement karke dekhte hain.


## ❌ Pehle dekho: BINA Decorator Pattern ke (Problem Code)

Chalo pehle ye dekhte hain ki Decorator Pattern KE BINA log Coffee Shop ke add-ons kaise
implement karte hain (subclassing se), aur usme kya dikkat aati hai.

In [2]:
"""
BINA DECORATOR KE — PROBLEM DEMO (SUBCLASSING SE)
--------------------------------------------------------
Yahan HAR combination ke liye ALAG SUBCLASS bana rahe hain.
Dekho neeche kya dikkat hai jab combinations badhte hain.
"""

class PlainCoffeeBad:
    def get_description(self):
        return "Plain Coffee"

    def get_cost(self):
        return 2.0


class CoffeeWithMilkBad(PlainCoffeeBad):
    def get_description(self):
        return "Plain Coffee, Milk"

    def get_cost(self):
        return 2.0 + 0.5


class CoffeeWithSugarBad(PlainCoffeeBad):
    def get_description(self):
        return "Plain Coffee, Sugar"

    def get_cost(self):
        return 2.0 + 0.2


# Ab agar customer "Milk + Sugar DONO" chahta hai... EK AUR NAYI CLASS banani padegi!
class CoffeeWithMilkAndSugarBad(PlainCoffeeBad):
    def get_description(self):
        return "Plain Coffee, Milk, Sugar"

    def get_cost(self):
        return 2.0 + 0.5 + 0.2


# Aur agar "Whipped Cream" bhi add karna ho (akele, ya milk ke saath, ya sugar ke saath,
# ya TEENO ke saath)... to KITNI AUR CLASSES banani padengi?
c1 = CoffeeWithMilkAndSugarBad()
print(f"{c1.get_description()} -> ${c1.get_cost()}")


Plain Coffee, Milk, Sugar -> $2.7


In [3]:
"""
ISME PROBLEM KYA THI?
-------------------------
1. CLASS EXPLOSION: Sirf 3 add-ons (Milk, Sugar, Cream) ke liye
   bhi POSSIBLE COMBINATIONS itni hain: Milk, Sugar, Cream,
   Milk+Sugar, Milk+Cream, Sugar+Cream, Milk+Sugar+Cream -- ye
   TO SIRF 3 add-ons ke saath 7 COMBINATIONS hain! Agar 6 add-ons
   ho gaye, to COMBINATIONS 63 tak pahunch jaayengi -- HAR EK ke
   liye ALAG CLASS?! Ye UNMANAGEABLE hai.

2. CODE DUPLICATION: "CoffeeWithMilkAndSugarBad" mein humne
   "2.0 + 0.5 + 0.2" MANUALLY LIKHA -- agar Milk ka price change
   ho (0.5 se 0.6), to YE CALCULATION HAR JAGAH DHOONDH KE UPDATE
   karni padegi jahan bhi Milk involve hai.

3. NO RUNTIME FLEXIBILITY: Agar customer order karte waqt DECIDE
   kare "abhi Sugar bhi chahiye", to hume EK BILKUL NAYI CLASS
   INSTANTIATE karni padegi -- EXISTING object mein DYNAMICALLY
   feature ADD NAHI kar sakte.

4. HARD TO MAINTAIN: Naya add-on (jaise "Chocolate") introduce
   karna ho, to EXISTING combinations ke liye BHI naye
   subclasses banane padenge (ChocolateMilkCoffee,
   ChocolateSugarCoffee, ...).

YE SAB PROBLEMS "Decorator Design Pattern" SOLVE KARTA HAI!
Neeche BEST APPROACH dekhte hain -- add-ons ko "WRAP" karenge,
jitne chaho utne COMBINE kar sakte ho, BINA naya CLASS banaye.
"""
print("Chalo ab BEST APPROACH (Decorator Pattern) dekhte hain...")


Chalo ab BEST APPROACH (Decorator Pattern) dekhte hain...


## ✅ Ab dekho: BEST APPROACH — Decorator Pattern se solve karte hain

### 1️⃣ Component Interface — `Coffee`

In [4]:
"""
COMPONENT INTERFACE (Coffee)
----------------------------------
Ye COMMON INTERFACE hai jise Concrete Component (PlainCoffee) AUR
Decorators DONO follow karenge. Isse Client donon ko SAME TAREEKE
se treat kar sakta hai -- use FARAK NAHI PADTA ki wo PLAIN coffee
hai ya WRAPPED coffee.
"""

from abc import ABC, abstractmethod

class Coffee(ABC):
    @abstractmethod
    def get_description(self):
        pass

    @abstractmethod
    def get_cost(self):
        pass


### 2️⃣ Concrete Component — `PlainCoffee`

In [5]:
"""
CONCRETE COMPONENT (PlainCoffee)
--------------------------------------
Ye BASE/CORE object hai -- iske paas KOI add-on nahi hai. Isi
object ko hum AAGE decorators se WRAP karenge.
"""

class PlainCoffee(Coffee):
    def get_description(self):
        return "Plain Coffee"

    def get_cost(self):
        return 2.0


### 3️⃣ Decorator (Abstract) — `CoffeeDecorator`

In [6]:
"""
DECORATOR (Abstract Wrapper)
----------------------------------
Ye bhi Coffee interface IMPLEMENT karta hai (isliye ye khud bhi
ek "Coffee" jaisa DIKHTA hai), lekin ANDAR ek "decorated_coffee"
reference rakhta hai -- jo WRAP kiya hua object hai.

DEFAULT behavior: WOHI return karo jo ANDAR wale object ne diya
(DELEGATE karo). Concrete Decorators isko OVERRIDE karke apna
EXTRA behavior add karenge.
"""

class CoffeeDecorator(Coffee):
    def __init__(self, decorated_coffee: Coffee):
        self.decorated_coffee = decorated_coffee

    def get_description(self):
        return self.decorated_coffee.get_description()

    def get_cost(self):
        return self.decorated_coffee.get_cost()


### 4️⃣ Concrete Decorators — `MilkDecorator`, `SugarDecorator`

In [7]:
"""
CONCRETE DECORATORS
------------------------
Har Concrete Decorator, CoffeeDecorator ko EXTEND karta hai aur
APNA EXTRA behavior add karta hai -- description mein NAAM JODTA
hai, aur cost mein APNA PRICE JODTA hai. BAAKI sab kaam ANDAR
wale "decorated_coffee" (parent) ke through DELEGATE hota hai.
"""

class MilkDecorator(CoffeeDecorator):
    def get_description(self):
        return self.decorated_coffee.get_description() + ", Milk"

    def get_cost(self):
        return self.decorated_coffee.get_cost() + 0.5


class SugarDecorator(CoffeeDecorator):
    def get_description(self):
        return self.decorated_coffee.get_description() + ", Sugar"

    def get_cost(self):
        return self.decorated_coffee.get_cost() + 0.2


class WhippedCreamDecorator(CoffeeDecorator):
    def get_description(self):
        return self.decorated_coffee.get_description() + ", Whipped Cream"

    def get_cost(self):
        return self.decorated_coffee.get_cost() + 0.7


### 5️⃣ Client — Add-ons ko RUNTIME par COMBINE karte hain

In [8]:
"""
CLIENT CODE
---------------
Ab dekho -- humein KISI NAYI CLASS ki ZAROORAT NAHI padi. Bas
PlainCoffee ko decorators se "WRAP" karte gaye, JITNI CHAHE
LAYERS lagate gaye!

MilkDecorator(PlainCoffee()) ka matlab hai:
    "PlainCoffee ko Milk se WRAP karo"

SugarDecorator(MilkDecorator(PlainCoffee())) ka matlab hai:
    "PEHLE Milk se wrap karo, PHIR us pure cheez ko Sugar se
    WRAP karo" -- LAYERS ki tarah!
"""

# Plain Coffee
coffee = PlainCoffee()
print(f"Description: {coffee.get_description()}")
print(f"Cost: ${coffee.get_cost()}")

# Coffee with Milk
milk_coffee = MilkDecorator(PlainCoffee())
print(f"\nDescription: {milk_coffee.get_description()}")
print(f"Cost: ${milk_coffee.get_cost()}")

# Coffee with Milk aur Sugar
sugar_milk_coffee = SugarDecorator(MilkDecorator(PlainCoffee()))
print(f"\nDescription: {sugar_milk_coffee.get_description()}")
print(f"Cost: ${sugar_milk_coffee.get_cost()}")

# Coffee with Milk, Sugar, AUR Whipped Cream -- bina koi NAYI class banaye!
full_loaded_coffee = WhippedCreamDecorator(SugarDecorator(MilkDecorator(PlainCoffee())))
print(f"\nDescription: {full_loaded_coffee.get_description()}")
print(f"Cost: ${full_loaded_coffee.get_cost()}")

print("\n✅ Dekha? Sirf 4 classes (Coffee, PlainCoffee, CoffeeDecorator, aur 3 Concrete Decorators) se HAR COMBINATION ban gaya -- bina class explosion ke!")


Description: Plain Coffee
Cost: $2.0

Description: Plain Coffee, Milk
Cost: $2.5

Description: Plain Coffee, Milk, Sugar
Cost: $2.7

Description: Plain Coffee, Milk, Sugar, Whipped Cream
Cost: $3.4000000000000004

✅ Dekha? Sirf 4 classes (Coffee, PlainCoffee, CoffeeDecorator, aur 3 Concrete Decorators) se HAR COMBINATION ban gaya -- bina class explosion ke!


## 📝 Practice Question (Tumhari baari!)

**Problem Statement:**

Ek **Pizza Order System** banao using Decorator Design Pattern (jaise document ke intro diagram
mein "Pizza + Capsicum + Cheese Burst" example diya gaya tha).

Ek Pizza ki BASE price hoti hai, aur customer optional TOPPINGS add kar sakta hai — har topping
ka apna EXTRA cost hota hai. Toppings ko JITNI CHAHO utni baar, KISI BHI ORDER mein COMBINE kar
sakte ho.

Tumhe ye banana hai (Coffee example jaisi hi structure use karo):

1. `Pizza` (Component interface) — abstract methods `get_description()` aur `get_cost()`
2. `PlainPizza` (Concrete Component) — description = `"Plain Pizza"`, cost = `5.0`
3. `PizzaDecorator` (abstract Decorator) — `decorated_pizza` reference rakhe, default delegate kare
4. Concrete Decorators: `CapsicumDecorator` (+$1.0), `CheeseBurstDecorator` (+$1.5), `OliveDecorator` (+$0.8)
5. Client code likho jo:
   - Sirf `PlainPizza` print kare
   - `PlainPizza` + `Capsicum` print kare
   - `PlainPizza` + `Capsicum` + `CheeseBurst` print kare
   - Sabhi TEENO toppings ke saath ek FULLY LOADED pizza print kare

👉 Pehle khud try karo neeche wale khaali code cell mein, phir uske baad wale cell mein solution dekho!

In [9]:
"""
YAHAN KHUD LIKHO (Practice Cell)
-----------------------------------
Hint: Coffee/PlainCoffee/CoffeeDecorator/MilkDecorator example ko
copy karke sirf naam badal do:
Coffee -> Pizza, PlainCoffee -> PlainPizza, CoffeeDecorator -> PizzaDecorator
MilkDecorator -> CapsicumDecorator, SugarDecorator -> CheeseBurstDecorator

Apna code neeche likho:
"""

# TODO: class Pizza (ABC) banao with abstract get_description() aur get_cost()
# TODO: class PlainPizza banao (cost=5.0)
# TODO: class PizzaDecorator banao (decorated_pizza reference, default delegate)
# TODO: CapsicumDecorator, CheeseBurstDecorator, OliveDecorator banao
# TODO: Client code likho jo alag alag combinations print kare


'\nYAHAN KHUD LIKHO (Practice Cell)\n-----------------------------------\nHint: Coffee/PlainCoffee/CoffeeDecorator/MilkDecorator example ko\ncopy karke sirf naam badal do:\nCoffee -> Pizza, PlainCoffee -> PlainPizza, CoffeeDecorator -> PizzaDecorator\nMilkDecorator -> CapsicumDecorator, SugarDecorator -> CheeseBurstDecorator\n\nApna code neeche likho:\n'

## ✅ Solution — Practice Question ka Coding Solution

In [10]:
"""
SOLUTION EXPLANATION
-----------------------
Yahan bilkul Coffee example jaisi hi structure use ki hai, bas ab
"Pizza" bana rahe hain instead of "Coffee".

1. Pizza               -> Component Interface
2. PlainPizza            -> Concrete Component
3. PizzaDecorator          -> Abstract Decorator
4. CapsicumDecorator,
   CheeseBurstDecorator,
   OliveDecorator          -> Concrete Decorators
5. Neeche client code hai jo alag alag combinations banata hai
"""

from abc import ABC, abstractmethod

# ---- 1. Component Interface ----
class Pizza(ABC):
    @abstractmethod
    def get_description(self):
        pass

    @abstractmethod
    def get_cost(self):
        pass


# ---- 2. Concrete Component ----
class PlainPizza(Pizza):
    def get_description(self):
        return "Plain Pizza"

    def get_cost(self):
        return 5.0


# ---- 3. Decorator (Abstract) ----
class PizzaDecorator(Pizza):
    def __init__(self, decorated_pizza: Pizza):
        self.decorated_pizza = decorated_pizza

    def get_description(self):
        return self.decorated_pizza.get_description()

    def get_cost(self):
        return self.decorated_pizza.get_cost()


# ---- 4. Concrete Decorators ----
class CapsicumDecorator(PizzaDecorator):
    def get_description(self):
        return self.decorated_pizza.get_description() + ", Capsicum"

    def get_cost(self):
        return self.decorated_pizza.get_cost() + 1.0


class CheeseBurstDecorator(PizzaDecorator):
    def get_description(self):
        return self.decorated_pizza.get_description() + ", Cheese Burst"

    def get_cost(self):
        return self.decorated_pizza.get_cost() + 1.5


class OliveDecorator(PizzaDecorator):
    def get_description(self):
        return self.decorated_pizza.get_description() + ", Olives"

    def get_cost(self):
        return self.decorated_pizza.get_cost() + 0.8


# ---- 5. Client Code ----
pizza = PlainPizza()
print(f"Description: {pizza.get_description()}")
print(f"Cost: ${pizza.get_cost()}")

capsicum_pizza = CapsicumDecorator(PlainPizza())
print(f"\nDescription: {capsicum_pizza.get_description()}")
print(f"Cost: ${capsicum_pizza.get_cost()}")

cheese_capsicum_pizza = CheeseBurstDecorator(CapsicumDecorator(PlainPizza()))
print(f"\nDescription: {cheese_capsicum_pizza.get_description()}")
print(f"Cost: ${cheese_capsicum_pizza.get_cost()}")

fully_loaded_pizza = OliveDecorator(CheeseBurstDecorator(CapsicumDecorator(PlainPizza())))
print(f"\nDescription: {fully_loaded_pizza.get_description()}")
print(f"Cost: ${fully_loaded_pizza.get_cost()}")


Description: Plain Pizza
Cost: $5.0

Description: Plain Pizza, Capsicum
Cost: $6.0

Description: Plain Pizza, Capsicum, Cheese Burst
Cost: $7.5

Description: Plain Pizza, Capsicum, Cheese Burst, Olives
Cost: $8.3


## 🎯 Bonus Challenge

Ab try karo:
1. Ek naya `MushroomDecorator` (+$1.2) add karo — kitni jagah EXISTING code change karna pada?
   (Answer: KAHIN NAHI! Sirf ek naya class banaya — ye Decorator Pattern ki taaqat hai)
2. Socho — agar customer ek topping do baar chahta hai (jaise "Extra Cheese Burst"), to kya
   `CheeseBurstDecorator(CheeseBurstDecorator(PlainPizza()))` likh kar ye possible hai? Try karke
   dekho!
3. Ek `get_toppings_count()` helper function likho jo bataye ki ek fully-wrapped pizza mein KITNI
   LAYERS (decorators) hain — is se samajh aayega ki debugging ke liye "layers count karna"
   Disadvantage section mein kyun mention hua tha.